In [1]:
import pandas as pd
from pyrosm import OSM
import folium
from folium.plugins import MarkerCluster
import gc

# **demand**

In [ ]:
# 1. กำหนดค่าเริ่มต้นและจำกัดพื้นที่เพื่อเซฟแรม
osm_filepath = "../../data/raw/thailand-260703.osm.pbf"
bkk_bbox = [100.30, 13.45, 100.95, 13.95]

print("1. 🚀 กำลังเปิดไฟล์ก้อนข้อมูลออฟไลน์และจำกัดพื้นที่กรุงเทพฯ...")
osm = OSM(osm_filepath, bounding_box=bkk_bbox)

print("2. 🏗️ กำลังสกัดสิ่งปลูกสร้างทั้งหมดในพื้นที่...")
buildings_gdf = osm.get_buildings()

print("3. 📌 กำลังถอดพิกัดตัวเลขเชิงพื้นที่จากจุดกึ่งกลาง (Centroid)...")
buildings_gdf['latitude'] = buildings_gdf.geometry.centroid.y
buildings_gdf['longitude'] = buildings_gdf.geometry.centroid.x

print("4. 📊 แปลงเป็น DataFrame และสกัดฟีเจอร์เพิ่มความลึกในการวิเคราะห์ (Feature Extraction)...")
df_flat = pd.DataFrame(buildings_gdf)

# ตรวจสอบคอลัมน์ที่จำเป็น ถ้าไม่มีในไฟล์ pbf ให้สร้างคอลัมน์ว่าง (NaN) ดักไว้ก่อนป้องกันโค้ดพัง
expected_features = {
    'name': 'Unknown',
    'building:levels': None,
    'building:flats': None,
    'addr:street': None,
    'addr:postcode': None
}
for col, default_val in expected_features.items():
    if col not in df_flat.columns:
        df_flat[col] = default_val

print("5. 🧹 กำลังกรองคัดแยกกลุ่มและทำความสะอาดข้อมูลข้อมูล...")
# กรองเอาเฉพาะตึกเป้าหมาย (Demand Proxies)
target_tags = ['apartments', 'residential', 'office', 'commercial', 'retail']
df_filtered = df_flat[df_flat['building'].isin(target_tags)].copy()

# ฟังก์ชันจัดกลุ่มประเภทตึกให้อ่านง่าย
def map_type(b_type):
    if b_type in ['apartments', 'residential']:
        return 'condo/apartment'
    elif b_type in ['office', 'commercial']:
        return 'office'
    elif b_type == 'retail':
        return 'mall'
    else:
        return 'others'


1. 🚀 กำลังเปิดไฟล์ก้อนข้อมูลออฟไลน์และจำกัดพื้นที่กรุงเทพฯ...
2. 🏗️ กำลังสกัดสิ่งปลูกสร้างทั้งหมดในพื้นที่...
3. 📌 กำลังถอดพิกัดตัวเลขเชิงพื้นที่จากจุดกึ่งกลาง (Centroid)...


/tmp/ipykernel_6353/2645082745.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  buildings_gdf['latitude'] = buildings_gdf.geometry.centroid.y
/tmp/ipykernel_6353/2645082745.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  buildings_gdf['longitude'] = buildings_gdf.geometry.centroid.x


4. 📊 แปลงเป็น DataFrame และสกัดฟีเจอร์เพิ่มความลึกในการวิเคราะห์ (Feature Extraction)...
5. 🧹 กำลังกรองคัดแยกกลุ่มและทำความสะอาดข้อมูลข้อมูล...


In [3]:
df_filtered['poi_type'] = df_filtered['building'].apply(map_type)

# อุดรอยรั่วตึกที่ไม่มีชื่อ (ถ้าไม่มีให้ใส่ประเภทตึกแทน)
df_filtered['name'] = df_filtered['name'].fillna(df_filtered['poi_type'])

# เลือกคอลัมน์ทั้งหมดแบบครบถ้วนสำหรับการทำโมเดลและแดชบอร์ดหลังบ้าน
df_final = df_filtered[[
    'name', 
    'poi_type', 
    'building:levels', 
    'building:flats', 
    'addr:street', 
    'addr:postcode', 
    'latitude', 
    'longitude'
]].copy()

# เปลี่ยนชื่อคอลัมน์ให้เป็นฟอร์แมตมาตรฐานของระบบ Database
df_final.columns = [
    'name', 
    'type', 
    'building_levels', 
    'building_flats', 
    'street', 
    'postcode', 
    'latitude', 
    'longitude'
]

# คลีนข้อมูลขั้นสุดท้าย: ลบจุดที่ไม่มีพิกัด และจุดพิกัดซ้ำกันเป๊ะๆ
df_final = df_final.dropna(subset=['latitude', 'longitude'])
df_final = df_final.drop_duplicates(subset=['latitude', 'longitude'])

print("\n--- 📊 🎉 สรุปยอดข้อมูลสำเร็จ (เวอร์ชันจัดเต็มครบทุกมิติ) ---")
print(df_final['type'].value_counts())
print(f"รวมข้อมูลเคลียร์สะอาดพร้อมใช้งาน: {len(df_final)} จุด")



--- 📊 🎉 สรุปยอดข้อมูลสำเร็จ (เวอร์ชันจัดเต็มครบทุกมิติ) ---
type
condo/apartment    13453
office              6072
mall                1195
Name: count, dtype: int64
รวมข้อมูลเคลียร์สะอาดพร้อมใช้งาน: 20720 จุด


In [ ]:
# 6. เซฟข้อมูลเป็นไฟล์ JSON ลงเครื่องถาวร
output_file = "../../data/interim/bangkok_pois.json"
df_final.to_json(output_file, orient='records', force_ascii=False, indent=4)
print(f"💾 บันทึกไฟล์สำเร็จ! ไฟล์เดตาสรุปหนาแน่นถูกเก็บไว้ที่: {output_file}")

# 7. 🧹 สั่ง Garbage Collection เคลียร์ RAM ก้อนยักษ์ทันทีเพื่อคืนพื้นที่ให้คอมพิวเตอร์
print("\n🧹 กำลังเคลียร์หน่วยความจำในเครื่องคอมพิวเตอร์...")
if 'buildings_gdf' in locals(): del buildings_gdf
if 'df_flat' in locals(): del df_flat
if 'df_filtered' in locals(): del df_filtered
gc.collect()
print("🤖 เคลียร์ RAM เรียบร้อย! คอมพิวเตอร์ของคุณกลับมาเบาหวิวแล้ว")

# 8. พล็อตแผนที่จำลองภาพรวมเพื่อทดสอบผลลัพธ์
print("\n🗺️ กำลังวาดแผนที่แสดงผล... (ดึง 2,000 จุดแรกมาพล็อตเพื่อความปลอดภัย)")
m = folium.Map(location=[13.7563, 100.5018], zoom_start=11)
marker_cluster = MarkerCluster().add_to(m)

color_map = {
    'mall': 'red',
    'office': 'blue',
    'condo/apartment': 'green'
}

for idx, row in df_final.head(2000).iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"<b>{row['name']}</b><br>ประเภท: {row['type']}<br>จำนวนชั้น: {row['building_levels']}",
        icon=folium.Icon(color=color_map.get(row['type'], 'gray'), icon='info-sign')
    ).add_to(marker_cluster)

print("🎉 แพลตฟอร์มพล็อตแมปปิ้งเสร็จสมบูรณ์ 100% แล้วครับ!")
m

💾 บันทึกไฟล์สำเร็จ! ไฟล์เดตาสรุปหนาแน่นถูกเก็บไว้ที่: bangkok_pois_clean.json

🧹 กำลังเคลียร์หน่วยความจำในเครื่องคอมพิวเตอร์...
🤖 เคลียร์ RAM เรียบร้อย! คอมพิวเตอร์ของคุณกลับมาเบาหวิวแล้ว

🗺️ กำลังวาดแผนที่แสดงผล... (ดึง 2,000 จุดแรกมาพล็อตเพื่อความปลอดภัย)
🎉 แพลตฟอร์มพล็อตแมปปิ้งเสร็จสมบูรณ์ 100% แล้วครับ!
